## Solution Code to Exercise (Problem for Maximizing Reward)

Here is the solution that codes the second problem we defined, where our overall goal was to maximize return while constraining our risk:


\begin{align*}
    \text{maximize } & \mathbf{m}^T\mathbf{w} \\
    \text{subject to the constraints } & \frac{1}{2} \mathbf{w}^T\mathbf{S}\mathbf{w}\leq\sigma^2_{max} \text{ and } \mathbf{1}^T\mathbf{w}=1
\end{align*}

**NOTE: This is the solution implemented with the changes that solve the issue of unboundedness identified in Exploration Task 2!**

In [3]:
import cvxpy as cp
import numpy as np
import yfinance as yf
import pandas as pd

def solve_markowitz(cov_matrix, expected_ror, max_variance):
    n = len(expected_ror)
    w = cp.Variable(n)

    # objective function
    objective = cp.Maximize(expected_ror @ w)

    # defining the constraints
    constraints = [
        cp.sum(w) == 1, # budget constraint: 1^T * w = 1
        0.5 * cp.quad_form(w, cov_matrix) <= max_variance, # variance constraint: 0.5 * w^T * S * w <= sigma^2_max
        w >= 0 # no short selling
      ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver = cp.CLARABEL)

    # debugging
    if prob.status not in ("optimal","optimal_inaccurate"):
      raise ValueError(f"Solver status: {prob.status} (no solution)")

    return w.value

# DRIVER CODE
# import data
tickers = ["AAPL", "MSFT", "GOOG","NVDA","META"]
data = yf.download(tickers,start = "2015-01-01",
                   end = "2025-01-01",
                   auto_adjust = False,
                   progress = False)
data = data["Adj Close"]
data = data.pct_change().dropna()

# sample covariance matrix
sample_cov_matrix = np.cov(m=data,rowvar=False)

# expected rates of return
sample_means = data.mean().values

# this is our sigma^2_max (maximum limit on 1/2 of the variance of our portfolio ROR)
# remember, our actual variance will be capped at 2 times the amount set here due to the 1/2 factor out front
max_variance = 0.0002

# finding our optimal weights
solution = solve_markowitz(cov_matrix=sample_cov_matrix,
                    expected_ror=sample_means,
                    max_variance=max_variance)

# printing our output
print("--- Optimal Portfolio Weights Using Markowitz MVO ---")
for name, wt in zip(data.columns, solution):
       print(f"{name} Weight: {wt*100:.4f}%")

# making sure our chosen weights satisfy the constraints
print(f"Total Weights (should equal 1): {np.sum(solution):.2f}")
print(f"Expected ROR on Portfolio Using Optimal Weights: {np.dot(solution, sample_means).item():.3e}")
portfolio_variance = np.dot(solution, sample_cov_matrix @ solution).item()
print(f"Portfolio Variance (should be less than or equal to 2 * {max_variance} = {2*max_variance}): {portfolio_variance:.3e}")

--- Optimal Portfolio Weights Using Markowitz MVO ---
AAPL Weight: 26.8440%
GOOG Weight: 7.8733%
META Weight: 0.0002%
MSFT Weight: 22.6293%
NVDA Weight: 42.6532%
Total Weights (should equal 1): 1.00
Expected ROR on Portfolio Using Optimal Weights: 1.765e-03
Portfolio Variance (should be less than or equal to 2 * 0.0002 = 0.0004): 4.000e-04
